# CUMULUS: Reproducible Preprocessing Benchmark

This notebook implements the **CUMULUS benchmark exactly as described in the paper**.

**Key properties**:
- Iterates over **all CSV files** in a given directory (e.g., 125 task-level files)
- Injects controlled corruption (MCAR missingness, spike outliers)
- Evaluates preprocessing stages **separately**:
  - Imputation (RMSE / MAE / Wilcoxon)
  - Outlier handling (variance reduction / KS)
  - Normalization (variance, skewness, kurtosis / KS)
- Aggregates metrics **across files** (median by default)
- Produces CSV result tables used for figures in the paper

> This notebook is intended to be run from a directory that contains the raw eye-tracking CSV files.


In [ ]:
# =========================
# Configuration (paper-aligned)
# =========================

import os
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.ensemble import IsolationForest

from scipy.stats import wilcoxon, ks_2samp, skew, kurtosis
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Path containing the 125 CSV files
DATA_DIR = "./data/raw"   # <- change if needed
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Evaluation features (paper)
FEATURES = ["Gaze X", "Gaze Y", "ET_PupilLeft", "ET_PupilRight"]

# Corruption levels
MISSING_LEVELS = [5, 10, 15, 20]
OUTLIER_LEVELS = [5, 10, 15, 20]

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# =========================
# Utility: load all CSV files
# =========================

def load_all_files(data_dir):
    files = sorted([
        os.path.join(data_dir, f)
        for f in os.listdir(data_dir)
        if f.endswith('.csv')
    ])
    assert len(files) > 0, "No CSV files found"
    print(f"Found {len(files)} CSV files")
    return files


In [ ]:
# =========================
# Corruption models (paper)
# =========================

def inject_mcar(df, col, pct):
    df = df.copy()
    n = int(len(df) * pct / 100)
    idx = np.random.choice(df.index, n, replace=False)
    df.loc[idx, col] = np.nan
    return df, idx

def inject_spike_outliers(df, col, pct):
    df = df.copy()
    n = int(len(df) * pct / 100)
    idx = np.random.choice(df.index, n, replace=False)
    df.loc[idx, col] += np.random.uniform(5, 10) * df[col].std()
    return df, idx


In [ ]:
# =========================
# Imputation methods (paper)
# =========================

def impute(df, method):
    if method == "mean":
        return pd.DataFrame(SimpleImputer(strategy="mean").fit_transform(df), columns=df.columns)
    if method == "locf":
        return df.fillna(method="ffill")
    if method == "knn":
        return pd.DataFrame(KNNImputer(n_neighbors=5).fit_transform(df), columns=df.columns)
    raise ValueError(method)


In [ ]:
# =========================
# Outlier handling (paper)
# =========================

def remove_outliers(series, method):
    x = series.values.reshape(-1, 1)
    mask = ~np.isnan(x).ravel()

    if method == "zscore":
        z = np.abs((x - np.nanmean(x)) / np.nanstd(x))
        x[z > 3] = np.nan

    elif method == "mad":
        med = np.nanmedian(x)
        mad = np.nanmedian(np.abs(x - med))
        x[np.abs(x - med) > 3 * mad] = np.nan

    elif method == "iforest":
        iso = IsolationForest(contamination=0.05, random_state=RANDOM_STATE)
        preds = iso.fit_predict(x[mask])
        x[mask][preds == -1] = np.nan

    return pd.Series(x.ravel(), index=series.index)


In [ ]:
# =========================
# Evaluation helpers (paper)
# =========================

def variance_reduction(x_orig, x_clean):
    return 1 - (np.nanvar(x_clean) / np.nanvar(x_orig))

def evaluate_imputation(x_orig, x_imp, mask):
    y_true = x_orig[mask]
    y_pred = x_imp[mask]
    return {
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
        "MAE": mean_absolute_error(y_true, y_pred),
        "Wilcoxon_p": wilcoxon(y_true, y_pred).pvalue
    }


## Main loop

The following cell performs the **full benchmark exactly as described in the paper**.
Results are aggregated across all files and written to CSV.


In [ ]:
files = load_all_files(DATA_DIR)

imputation_results = []

for file in files:
    df0 = pd.read_csv(file)
    df0 = df0[FEATURES].dropna()

    for lvl in MISSING_LEVELS:
        for col in FEATURES:
            corrupted, idx = inject_mcar(df0[[col]], col, lvl)
            mask = idx

            for method in ["mean", "locf", "knn"]:
                imputed = impute(corrupted, method)
                metrics = evaluate_imputation(df0[col], imputed[col], mask)

                imputation_results.append({
                    "file": os.path.basename(file),
                    "feature": col,
                    "missing_pct": lvl,
                    "method": method,
                    **metrics
                })

imputation_df = pd.DataFrame(imputation_results)
imputation_df.to_csv(os.path.join(RESULTS_DIR, "imputation_results_raw.csv"), index=False)

summary = imputation_df.groupby(["feature", "missing_pct", "method"]).median().reset_index()
summary.to_csv(os.path.join(RESULTS_DIR, "imputation_results_summary.csv"), index=False)

summary.head()